# Semana 6 — Del mundo a un modelo computacional: entidades, estado y dinámica

**Electiva Profesional I (El Metaverso) · 2026-II**

La semana pasada consolidó la galería en `escena_base`: un diccionario con entidades, atributos y un estado — pero un estado **quieto**, congelado en un instante. Hoy ese estado empieza a **cambiar**: una regla que se aplica una y otra vez, en pasos de tiempo discretos y que mueve o transforma las entidades del mundo. Eso es una **dinámica** — el corazón de la Unidad 2.

Con esa idea ya construida sobre la galería, usted define **su propio universo** (el del proyecto): sus entidades, su estado inicial y una dinámica propia. Cierra subiendo ese avance a un repositorio remoto.

**Aviso para hoy:** en la Parte 2 hay una celda que va a fallar con `NameError` la primera vez que la corra. Es a propósito — el valor que la completa se anuncia en clase, en el Punto de encuentro y no está escrito en otra parte.

---
## 0. Entorno

Mismo entorno `mv2026` de las semanas 1 a 5.

In [1]:
import sys
import numpy as np
import copy

print('Ejecutable (debe mencionar mv2026):', sys.executable)
print('numpy:', np.__version__)

try:
    import pythreejs as p3
    TENGO_3D = True
    print('pythreejs: OK')
except Exception as e:
    TENGO_3D = False
    print('pythreejs: no disponible ->', e)


Ejecutable (debe mencionar mv2026): C:\Users\loghan\miniconda3\envs\mv2026\python.exe
numpy: 2.5.1
pythreejs: OK


---
## 1. Recuperar `escena_base`

Sin predicción todavía: recupere la estructura consolidada de la semana 5, con los mismos números ya verificados.

In [2]:
escena_base = {
    'sala': {
        'tipo': 'piso',
        'posicion': [5.0, 0.0, 0.0],
        'dimensiones': {'width': 40, 'height': 20},
    },
    'pared': {
        'posicion': [0.0, 2.0, -4.0],
        'dimensiones': {'width': 14, 'height': 4, 'depth': 0.2},
    },
    'obras': [
        {'nombre': 'El Reflejo Digital', 'posicion_original': [2.0, 0.0, 5.0],
         'posicion_actual': [13.0, 0.0, 2.5], 'color': '#00ff00'},
        {'nombre': 'Fragmentos', 'posicion_original': [-3.0, 1.5, 2.0],
         'posicion_actual': [10.5, 0.75, 1.0], 'color': '#ff5500'},
        {'nombre': 'Eco', 'posicion_original': [5.0, 0.0, 1.0],
         'posicion_actual': [14.5, 0.0, 0.5], 'color': '#0055ff'},
    ],
}

destino_sala = np.array([12.0, 0.0, 0.0])  # centro de la sala, ya usado en la semana 5

print('obras en escena_base:', [o['nombre'] for o in escena_base['obras']])


obras en escena_base: ['El Reflejo Digital', 'Fragmentos', 'Eco']


---
## 2. De una posición fija a una posición que cambia con el tiempo

`escena_base` guarda una **sola** posición por obra — un instante congelado. Para que una obra **se mueva de verdad**, hace falta una regla que, a partir de la posición de ahora, calcule la posición del **próximo instante** (lo que en simulación se llama un **paso de tiempo** o *tick*).

Vamos a hacer que **"El Reflejo Digital"** orbite alrededor del centro de la sala (`destino_sala`), rotando un ángulo fijo en cada paso, sobre el plano horizontal (los ejes X y Z; la altura Y no cambia).

Prediga antes de correr: la posición actual de "El Reflejo Digital" es `[13.0, 0.0, 2.5]` y el centro de la sala es `[12.0, 0.0, 0.0]`. Si la obra rota alrededor del centro ¿Su distancia al centro **cambia** de un paso a otro o se mantiene igual?

_(reemplace esta línea)_ 

Se mantiene igual, justamente porque la orbita es circular

In [3]:
def rota_alrededor_del_centro(posicion, centro, angulo):
    """Rota 'posicion' un 'angulo' (radianes) alrededor de 'centro', sobre el plano X-Z."""
    relativo = np.array(posicion) - centro
    x, y, z = relativo
    x_nueva = x * np.cos(angulo) - z * np.sin(angulo)
    z_nueva = x * np.sin(angulo) + z * np.cos(angulo)
    return centro + np.array([x_nueva, y, z_nueva])

# Un solo paso, de demostracion, con un angulo cualquiera (15 grados) solo para ver el
# mecanismo.
reflejo = next(o for o in escena_base['obras'] if o['nombre'] == 'El Reflejo Digital')
un_paso = rota_alrededor_del_centro(reflejo['posicion_actual'], destino_sala, np.pi / 12)
print('posicion actual:      ', np.array(reflejo['posicion_actual']).round(3))
print('despues de un paso:    ', un_paso.round(3))

distancia_antes = np.linalg.norm(np.array(reflejo['posicion_actual']) - destino_sala)
distancia_despues = np.linalg.norm(un_paso - destino_sala)
print('distancia al centro, antes:  ', round(distancia_antes, 3))
print('distancia al centro, despues:', round(distancia_despues, 3))


posicion actual:       [13.   0.   2.5]
despues de un paso:     [12.319  0.     2.674]
distancia al centro, antes:   2.693
distancia al centro, despues: 2.693


### 🅢 Punto de encuentro — *(pausa guiada)*

- **Dinámica** — una regla que calcula el estado del **próximo instante** a partir del estado **ahora**. No es una posición nueva escrita a mano: es una función que se puede aplicar una y otra vez.
- **Paso de tiempo (*tick*)** — cada aplicación de esa regla. Tres pasos son tres aplicaciones seguidas, cada una partiendo del resultado de la anterior.
- **Regla de actualización** — el nombre técnico de la función que implementa la dinámica (`rota_alrededor_del_centro`, en el ejemplo de hoy).
- Note que la **distancia al centro no cambió** — eso es lo que hace que rotar sea distinto de, por ejemplo, alejarse: la dinámica de hoy conserva una cantidad (la distancia) mientras cambia otra (el ángulo).

*"Una posición fija es una fotografía. Una dinámica es una película: la regla que genera el siguiente cuadro a partir del actual."*

---
## 3. Simule varios pasos — la celda que falta un dato

Para simular varios pasos seguidos, hace falta fijar **cuánto rota la obra en cada paso**. Ese ángulo no está escrito en este cuaderno — el docente lo anuncia en clase, en el Punto de encuentro de hoy.

**La celda de abajo va a fallar con `NameError` la primera vez que la corra — es intencional.** Corra la celda ahora para ver el error, documéntelo y vuelva después del anuncio para completarla.

In [4]:
ANGULO_PASO = np.pi / 6  # el docente anuncia el incremento angular por paso en clase

posicion = np.array(reflejo['posicion_actual'])
for paso in range(1, 4):
    posicion = rota_alrededor_del_centro(posicion, destino_sala, ANGULO_PASO)
    print(f'paso {paso}: {posicion.round(3)}')

assert np.allclose(posicion, [9.5, 0.0, 1.0], atol=0.05), (
    f'despues de 3 pasos se esperaba una posicion cercana a [9.5, 0.0, 1.0], dio {posicion.round(3)}'
)
print('Checkpoint superado: tres pasos de la dinamica dan la posicion esperada.')


NameError: name '____' is not defined

**`[Trabajo del estudiante]`** Una vez que reemplazó `____` por el ángulo anunciado y el checkpoint pasó: ¿Qué pasaría con la trayectoria de la obra si `ANGULO_PASO` fuera negativo en vez de positivo? No lo corra — razone a partir de lo que ya vio sobre la distancia al centro y la dirección del giro.

_(reemplace esta línea)_

Si ANGULO_PASO fuera negativo, la trayectoria mantendría exactamente el mismo radio y distancia al centro, pero la obra rotaría en sentido horario (a favor de las manecillas del reloj) en lugar de antihorario. La figura recorrería el mismo arco circular, simplemente invirtiendo la dirección de su movimiento a lo largo del plano X-Z.

---
## 4. Su propio universo: banco de mini-casos

La galería fue el ejemplo compartido. Desde aquí, el mundo es el **suyo** — el del proyecto.

Elija **uno** de estos mini-casos como punto de partida (o proponga el suyo, si ya tiene una idea clara, siempre que sea un dominio neutral):

1. **Un mercado simple** — puestos, compradores, precios que suben o bajan según la demanda.
2. **Un ecosistema** — depredadores, presas y un recurso que se repone con el tiempo.
3. **Una multitud en una plaza** — personas que se mueven hacia salidas, con posible aglomeración.
4. **El tránsito de una intersección** — vehículos, semáforos y una cola que crece o se vacía.
5. **Un mundo de juego neutral** — jugadores, objetos recolectables y un puntaje que cambia.

Antes de escribir código: en una frase ¿Qué **decisión** apoyaría simular este mundo? (por ejemplo: "decidir cuántos puestos abrir", "decidir si una salida alcanza para evacuar a tiempo"). Un mundo simulado sin una decisión detrás es una animación, no un modelo — la misma distinción que ya construyó con la línea base y la verificación en semanas anteriores.

_(reemplace esta línea)_ 

Decidir la duración óptima del semáforo en verde para evitar que la cola de vehículos supere la capacidad máxima del carril y congestione la intersección.

In [4]:
import copy
import numpy as np

mi_mundo = {
    'interseccion': {
        'semaforo_verde': True,
        'tiempo_cambio': 5
    },
    'entidades': [
        {'nombre': 'carril_norte', 'longitud_cola': 12, 'tasa_llegada': 3, 'tasa_salida': 4},
        {'nombre': 'carril_este', 'longitud_cola': 5, 'tasa_llegada': 2, 'tasa_salida': 4}
    ]
}


def mi_regla_dinamica(mundo):
    """Recibe el estado actual y devuelve un NUEVO diccionario: el estado un paso despues.

    No modifique 'mundo' directamente -- construya y devuelva una copia
    actualizada.
    """
    nuevo_mundo = copy.deepcopy(mundo)

    # Actualizar la cola de cada carril en un paso de tiempo
    for carril in nuevo_mundo['entidades']:
        # Llegan nuevos vehículos siempre
        llegadas = carril['tasa_llegada']

        # Solo salen vehículos si el semáforo está en verde para ese flujo
        if nuevo_mundo['interseccion']['semaforo_verde']:
            salidas = carril['tasa_salida'] if carril['nombre'] == 'carril_norte' else 0
        else:
            salidas = carril['tasa_salida'] if carril['nombre'] == 'carril_este' else 0

        # La cola se actualiza (nunca puede ser menor a 0)
        carril['longitud_cola'] = max(0, carril['longitud_cola'] + llegadas - salidas)

    return nuevo_mundo


print(mi_mundo)


{'interseccion': {'semaforo_verde': True, 'tiempo_cambio': 5}, 'entidades': [{'nombre': 'carril_norte', 'longitud_cola': 12, 'tasa_llegada': 3, 'tasa_salida': 4}, {'nombre': 'carril_este', 'longitud_cola': 5, 'tasa_llegada': 2, 'tasa_salida': 4}]}


In [5]:
# Autochequeo: esta prueba es la misma para cualquier mundo -- no depende de que dominio eligio.
assert isinstance(mi_mundo, dict) and len(mi_mundo) > 0, 'mi_mundo debe ser un diccionario no vacio'

estado_antes = copy.deepcopy(mi_mundo)
estado_despues = mi_regla_dinamica(mi_mundo)

assert isinstance(estado_despues, dict), 'mi_regla_dinamica debe devolver un diccionario'
assert estado_despues != estado_antes, (
    'el estado despues de un paso es IGUAL al de antes -- una dinamica que no cambia nada '
    'no es una dinamica real. Revise que mi_regla_dinamica este actualizando algun valor.'
)
assert mi_mundo == estado_antes, (
    'mi_mundo cambio despues de llamar a mi_regla_dinamica -- la funcion debe devolver una '
    'copia nueva, no modificar el mundo original en el sitio.'
)
print('Checkpoint superado: su dinamica produce un estado nuevo y distinto, sin mutar el original.')
print('estado antes:  ', estado_antes)
print('estado despues:', estado_despues)


Checkpoint superado: su dinamica produce un estado nuevo y distinto, sin mutar el original.
estado antes:   {'interseccion': {'semaforo_verde': True, 'tiempo_cambio': 5}, 'entidades': [{'nombre': 'carril_norte', 'longitud_cola': 12, 'tasa_llegada': 3, 'tasa_salida': 4}, {'nombre': 'carril_este', 'longitud_cola': 5, 'tasa_llegada': 2, 'tasa_salida': 4}]}
estado despues: {'interseccion': {'semaforo_verde': True, 'tiempo_cambio': 5}, 'entidades': [{'nombre': 'carril_norte', 'longitud_cola': 11, 'tasa_llegada': 3, 'tasa_salida': 4}, {'nombre': 'carril_este', 'longitud_cola': 7, 'tasa_llegada': 2, 'tasa_salida': 4}]}


**`[Trabajo del estudiante]`** Con su dinámica ya funcionando: ¿Qué atributo cambia entre `estado_antes` y `estado_despues` y por qué ese es justamente el atributo que le interesa observar para la decisión que escribió arriba? Si el atributo que cambia no tiene relación con esa decisión, es una señal de que vale la pena revisar el modelo antes de seguir.

_(reemplace esta línea)_

El atributo que cambia es longitud_cola de cada carril. Este atributo es clave porque mide directamente la acumulación de vehículos en la intersección; observando si la cola crece o disminuye en cada paso de tiempo se evalúa si el tiempo de luz verde es suficiente para descongestionar el tráfico o si causará un trancon.

---
## 5. Git remoto: suba su avance

Hasta ahora su repositorio vive solo en su computador (Git local, semana 5). Hoy lo conecta a un repositorio remoto para que el avance quede respaldado y visible.

**En PowerShell, no en este cuaderno:**

1. Cree un repositorio **privado y vacío** en `github.com/new` (sin README, sin `.gitignore`, sin licencia — para que no choque con su historial local ya existente).
2. `git remote add origin <URL-de-su-repositorio>`.
3. Confirme que este cuaderno (con `mi_mundo` y `mi_regla_dinamica` ya escritos) está guardado, y agréguelo: `git add .` (revise con `git status` que `.ipynb_checkpoints/` y `__pycache__/` **no** aparezcan — su `.gitignore` de la semana 5 debería evitarlo).
4. `git commit -m "avance del proyecto: entidades, estado y dinamica"`.
5. `git push -u origin main` (o `master`, según el nombre que Git le haya dado a su rama local — revíselo con `git branch` si no está seguro).

> **El momento "no da error pero está mal" de esta semana.** Si guarda el cuaderno con `mi_mundo` ya escrito pero **olvida correr `git add` sobre el archivo actualizado** antes del commit, `git commit` y `git push` van a funcionar **sin quejarse** — pero van a subir la versión vieja del cuaderno, sin su avance. `git push` que "funciona" no es lo mismo que `git push` que subió lo que usted cree que subió; por eso la siguiente celda pide la evidencia real, no la promesa de que ya se hizo.

In [7]:
# [Trabajo del estudiante] ejecute esta celda tal cual, despues de conectar el repositorio remoto.
!git remote -v


origin	https://github.com/ejemplo/ejemplo.git (fetch)
origin	https://github.com/ejemplo/ejemplo.git (push)


---
## Cierre del martes

Guarde este cuaderno con `mi_mundo`, `mi_regla_dinamica` y el `ANGULO_PASO` ya resueltos. El miércoles retoma exactamente aquí, para confirmar que el `push` remoto realmente subió su avance.

---
## 6. Miércoles — evidencia del `push` remoto

Ejecute estas dos celdas **desde este mismo cuaderno**, ya con su `push` hecho en PowerShell. No copie un resultado de otra persona: si el historial no es el suyo, no cuenta como evidencia.

In [8]:
# [Trabajo del estudiante] ejecute esta celda tal cual, despues de su push.
!git log --oneline


33a33b3 (HEAD -> master) primer commit: material de prueba


In [9]:
# [Trabajo del estudiante] ejecute esta celda tal cual. Confirme que no hay cambios sin
# confirmar relacionados con este cuaderno (si los hay, su push no incluyo la version final).
!git status


On branch master
nothing to commit, working tree clean


---
## 7. Informe de cierre

En cuatro a seis frases: ¿qué mundo eligió y por qué esa decisión concreta le pareció interesante de apoyar con una simulación? ¿Qué fue lo más confuso de escribir su propia regla de dinámica, comparado con seguir la de la rotación de la galería? Si le pasó el problema de "olvidar el `git add` antes del commit", cuéntelo; si no, diga cómo lo evitó.

_(reemplace esta línea con su respuesta)_